# Notebook 03 — Visualisation Multidimensionnelle

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook produit des **graphiques multidimensionnels percutants** pour communiquer les insights cles aux equipes pedagogiques. Chaque visualisation est construite pour etre directement utilisable dans un rapport ou un tableau de bord metier.

Graphiques produits :
1. Profils de risque par programme et semestre
2. Impact du statut boursier et de l'acces internet
3. Heatmap des correlations d'apprentissage
4. Distributions comparatives (violin plots)
5. Scatter plot multidimensionnel (engagement vs. notes)
6. Portrait-robot de l'etudiant a risque

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from utils_viz import set_custom_style, plot_correlation_matrix, plot_bivariate_scatter

set_custom_style(theme='light')
%matplotlib inline

df = pd.read_csv('data/processed/tp1_student_risk_wrangled.csv')
df['dropout_risk'] = df['dropout_risk'].astype(bool)
df['scholarship'] = df['scholarship'].astype(bool)
df['Statut'] = df['dropout_risk'].map({True: 'A risque', False: 'Non a risque'})

PALETTE = {'Non a risque': '#188038', 'A risque': '#D93025'}
print(f'Donnees chargees : {df.shape}  |  Taux de risque : {df["dropout_risk"].mean()*100:.1f} %')

## Visualisation 1 — Profils de risque par programme et semestre

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Taux de risque par programme (trie decroissant)
prog_risk = df.groupby('program')['dropout_risk'].mean() * 100
prog_risk = prog_risk.sort_values(ascending=True)
colors = ['#188038' if v < 7 else '#F29900' if v < 10 else '#D93025' for v in prog_risk.values]
axes[0].barh(prog_risk.index, prog_risk.values, color=colors, edgecolor='none', height=0.55)
axes[0].set_xlabel('Taux de risque (%)')
axes[0].set_title('Taux d\'abandon par programme')
for i, v in enumerate(prog_risk.values):
    axes[0].text(v + 0.15, i, f'{v:.1f}%', va='center', fontsize=9, fontweight='bold')
axes[0].set_xlim(0, prog_risk.max() * 1.3)

# Evolution du risque par semestre
sem = df.groupby('semester')['dropout_risk'].mean() * 100
axes[1].plot(sem.index, sem.values, 'o-', color='#1A73E8', linewidth=2.5, markersize=8)
axes[1].fill_between(sem.index, sem.values, alpha=0.1, color='#1A73E8')
axes[1].set_xlabel('Semestre')
axes[1].set_ylabel('Taux de risque (%)')
axes[1].set_title('Evolution du risque au fil des semestres')
axes[1].set_xticks(sem.index)
for x, y in zip(sem.index, sem.values):
    axes[1].annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)

fig.suptitle('Insight 1 & 2 : Risque concentre sur Digital Design et aux semestres 2 & 6',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('report/assets/tp2_program_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 2 — Impact du statut boursier et de l'acces internet

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Statut boursier
bourse_risk = df.groupby('scholarship')['dropout_risk'].mean() * 100
labels_b = ['Non-boursier', 'Boursier']
vals_b = [bourse_risk.get(False, 0), bourse_risk.get(True, 0)]
bars0 = axes[0].bar(labels_b, vals_b, color=['#D93025', '#188038'], edgecolor='none', width=0.45)
axes[0].set_ylabel('Taux de risque (%)')
axes[0].set_title('Taux de risque selon le statut boursier')
axes[0].bar_label(bars0, fmt='%.1f %%', fontsize=11, fontweight='bold', padding=3)
axes[0].set_ylim(0, max(vals_b) * 1.35)

# Acces internet
inet_risk = df.groupby('internet_access')['dropout_risk'].mean() * 100
labels_i = ['Sans internet', 'Avec internet']
vals_i = [inet_risk.get(False, 0), inet_risk.get(True, 0)]
bars1 = axes[1].bar(labels_i, vals_i, color=['#D93025', '#188038'], edgecolor='none', width=0.45)
axes[1].set_ylabel('Taux de risque (%)')
axes[1].set_title('Taux de risque selon l\'acces internet')
axes[1].bar_label(bars1, fmt='%.1f %%', fontsize=11, fontweight='bold', padding=3)
axes[1].set_ylim(0, max(vals_i) * 1.35)

fig.suptitle('Insight 3 : Le statut boursier et l\'acces internet jouent un role protecteur',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('report/assets/tp2_student_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 3 — Heatmap des correlations

In [ ]:
learning_cols = [
    'study_hours_per_week', 'lms_sessions_week', 'attendance_rate',
    'assignment_delay_days', 'prior_average', 'continuous_assessment',
    'stress_index', 'engagement_score'
]
fig_corr = plot_correlation_matrix(df, learning_cols)
fig_corr.axes[0].set_title('Matrice de correlation des variables d\'apprentissage', fontsize=12)
fig_corr.savefig('report/assets/tp2_learning_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

corr = df[learning_cols].corr()
corr_abs = corr.abs()
np.fill_diagonal(corr_abs.values, 0)
best = corr_abs.stack().idxmax()
print(f'Paire la plus correlee : {best[0]} <-> {best[1]} : r = {corr.loc[best]:.3f}')

## Visualisation 4 — Distributions comparatives (violin plots)

In [ ]:
feature_cols = [
    'attendance_rate', 'prior_average', 'continuous_assessment',
    'assignment_delay_days', 'engagement_score', 'academic_pressure_index'
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, col in zip(axes.flat, feature_cols):
    sns.violinplot(
        data=df, x='Statut', y=col,
        palette=PALETTE, ax=ax,
        inner='box', linewidth=1
    )
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('')

fig.suptitle('Insight 4 & 5 : Distribution des variables cles selon le risque', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Visualisation 5 — Scatter plot multidimensionnel

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for risk_val, label, color, marker in [
    (False, 'Non a risque', '#188038', 'o'),
    (True, 'A risque', '#D93025', 'X')
]:
    subset = df[df['dropout_risk'] == risk_val]
    ax.scatter(
        subset['engagement_score'],
        subset['prior_average'],
        c=color, marker=marker, alpha=0.5, s=30,
        label=label, edgecolors='none'
    )

ax.set_xlabel('Score d\'engagement')
ax.set_ylabel('Moyenne anterieure (/20)')
ax.set_title('Engagement vs. Performance academique selon le risque')
ax.legend()
fig.tight_layout()
fig.savefig('report/assets/tp2_financial_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 6 — Portrait-robot de l'etudiant a risque

In [ ]:
# Comparaison des moyennes : a risque vs. non a risque
compare_cols = [
    'attendance_rate', 'study_hours_per_week', 'lms_sessions_week',
    'prior_average', 'continuous_assessment', 'assignment_delay_days',
    'stress_index', 'commute_minutes'
]
mean_risk = df[df['dropout_risk'] == True][compare_cols].mean()
mean_no_risk = df[df['dropout_risk'] == False][compare_cols].mean()

# Normalisation pour le graphique en radar / barplot
comparison = pd.DataFrame({
    'A risque': mean_risk,
    'Non a risque': mean_no_risk,
    'Ecart (%)': ((mean_risk - mean_no_risk) / mean_no_risk * 100).round(1)
})

print('=== Portrait-robot : moyennes comparees ===')
print(comparison.round(2).to_string())

In [ ]:
# Barplot des ecarts relatifs
ecarts = comparison['Ecart (%)'].sort_values()
colors_e = ['#D93025' if v > 0 else '#188038' for v in ecarts.values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ecarts.index, ecarts.values, color=colors_e, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Ecart relatif (%) : etudiant a risque vs. non a risque')
ax.set_title('Portrait-robot de l\'etudiant a risque — Ecarts relatifs aux moyennes')
for i, v in enumerate(ecarts.values):
    offset = 0.5 if v >= 0 else -0.5
    ax.text(v + offset, i, f'{v:+.1f}%', va='center', fontsize=9)

red = mpatches.Patch(color='#D93025', label='Facteur aggravant')
green = mpatches.Patch(color='#188038', label='Facteur protecteur')
ax.legend(handles=[red, green])

fig.tight_layout()
fig.savefig('report/assets/tp2_transport_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Synthese des insights visuels

| # | Visualisation | Insight principal |
|---|---|---|
| 1 | Profils par programme | `Digital Design` : 12.6 % de risque vs. 4.4 % pour `Data Science` |
| 2 | Evolution par semestre | Pics de risque aux semestres 2 et 6 (points de bascule academique) |
| 3 | Statut boursier | Les boursiers ont 2x moins de risque (~5 % vs. ~11 %) |
| 4 | Heatmap correlations | `prior_average` / `continuous_assessment` : r = 0.756 (stabilite du niveau) |
| 5 | Portrait-robot | Les etudiants a risque ont +30 % de retard, -8 % d'assiduite, +12 % de stress |